<a href="https://colab.research.google.com/github/AdityaJare/ML-PRojects/blob/main/bank_marketing_ann_vs_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
!pip install keras-tuner

In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import tensorflow
from tensorflow import keras
import keras_tuner as kt
from tensorflow.keras import layers


In [65]:
df = pd.read_csv("/content/bank-additional-full.csv",sep=";")

In [66]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [68]:
df.describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,40.02406,258.285010,2.567593,962.475454,0.172963,0.081886,93.575664,-40.502600,3.621291,5167.035911
std,10.42125,259.279249,2.770014,186.910907,0.494901,1.570960,0.578840,4.628198,1.734447,72.251528
min,17.00000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.00000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.00000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.00000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.00000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [69]:
df.isnull().sum()

,0
age,0
job,0
marital,0
education,0
default,0
housing,0
loan,0
contact,0
month,0
day_of_week,0


In [70]:
df['y'].value_counts()

,count
y,
no,36548
yes,4640


In [71]:
df['previously_contacted'] = (df['pdays'] != 999).astype(int)
df= df.drop(columns="pdays")
df = df.drop(columns="duration")

In [72]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,previously_contacted
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0


In [73]:
df['y'] = df['y'].map({'yes': 1, 'no': 0})

In [74]:
for col in df.select_dtypes(include='object').columns:
    print(col, df[col].unique())

job ['housemaid' 'services' 'admin.' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']
marital ['married' 'single' 'divorced' 'unknown']
education ['basic.4y' 'high.school' 'basic.6y' 'basic.9y' 'professional.course'
 'unknown' 'university.degree' 'illiterate']
default ['no' 'unknown' 'yes']
housing ['no' 'yes' 'unknown']
loan ['no' 'yes' 'unknown']
contact ['telephone' 'cellular']
month ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'mar' 'apr' 'sep']
day_of_week ['mon' 'tue' 'wed' 'thu' 'fri']
poutcome ['nonexistent' 'failure' 'success']


In [75]:
df['job'].value_counts()

,count
job,
admin.,10422
blue-collar,9254
technician,6743
services,3969
management,2924
retired,1720
entrepreneur,1456
self-employed,1421
housemaid,1060


In [76]:
df["marital"].value_counts()

,count
marital,
married,24928
single,11568
divorced,4612
unknown,80


In [77]:
df["education"].value_counts()

,count
education,
university.degree,12168
high.school,9515
basic.9y,6045
professional.course,5243
basic.4y,4176
basic.6y,2292
unknown,1731
illiterate,18


In [78]:
df["default"].value_counts()

,count
default,
no,32588
unknown,8597
yes,3


In [79]:
df["housing"].value_counts()

,count
housing,
yes,21576
no,18622
unknown,990


In [80]:
df["loan"].value_counts()

,count
loan,
no,33950
yes,6248
unknown,990


In [81]:
import numpy as np
from sklearn.impute import SimpleImputer

impute_cols = ['job', 'marital', 'education']
df[impute_cols] = df[impute_cols].replace('unknown', np.nan)

In [82]:
print(df[['job', 'marital', 'education']].isnull().sum())

job           330
marital        80
education    1731
dtype: int64


In [83]:
onehot_impute_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

education_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=[[
        'illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
        'high.school', 'professional.course', 'university.degree'
    ]]))
])

scaling_pipe = Pipeline([("scaling", StandardScaler())])

onehot_keep_pipe = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [84]:
numeric_cols = ['age', 'campaign', 'previous', 'previously_contacted',
                'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
                'euribor3m', 'nr.employed']
onehot_impute_cols = ['job', 'marital']
onehot_keep_cols = ['default', 'housing', 'loan', 'contact',
                    'poutcome', 'month', 'day_of_week']
education_col = ['education']

In [85]:
preprocessing = ColumnTransformer([
    ("scaling", scaling_pipe, numeric_cols),
    ("onehot_impute", onehot_impute_pipe, onehot_impute_cols),
    ("onehot_keep", onehot_keep_pipe, onehot_keep_cols),
    ("education", education_pipe, education_col),
])

In [86]:
X = df.drop("y", axis=1)
Y = df["y"]

assigned = numeric_cols + onehot_impute_cols + onehot_keep_cols + education_col
print(set(X.columns) - set(assigned))
print(len(assigned), len(X.columns))

set()
19 19


In [87]:
X.shape

(41188, 19)

In [88]:
x_train, x_test,y_train, y_test =  train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

x_train_preprocessed = preprocessing.fit_transform(x_train)
x_test_preprocessed = preprocessing.transform(x_test)

In [89]:
n_features = x_train_preprocessed.shape[1]

In [90]:
n_features

53

In [94]:
def build_model(hp):

  model=keras.Sequential()

  model.add(layers.Input(shape=(n_features,)))

  for i in range(hp.Int("num_layers", 1, 3)):
    model.add(layers.Dense(
        units=hp.Int(f"unit_{i}", min_value=16, max_value=128, step=16),
        activation=hp.Choice(f"activation_{i}", values=["relu", "sigmoid", "tanh"])
    ))
    model.add(layers.Dropout(hp.Float(f"dropout_{i}", min_value=0.1, max_value=0.5, step=0.1)))

  model.add(layers.Dense(1, activation="sigmoid"))

  lr=hp.Choice(f"learning_rate", values=[0.01, 0.001, 0.0001])
  opt_name = hp.Choice(f"optimizer", values=["adam", "rmsprop", "nadam"])
  optimizer = {
      "adam"  : keras.optimizers.Adam(learning_rate=lr),
      "rmsprop" : keras.optimizers.RMSprop(learning_rate=lr),
      "nadam" : keras.optimizers.Nadam(learning_rate=lr)
  }[opt_name]

  model.compile(
      optimizer=optimizer,
      loss="binary_crossentropy",
      metrics=["accuracy", keras.metrics.AUC(name="auc")]
  )

  return model

In [95]:
test_model = build_model(kt.HyperParameters())
test_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 881 (3.44 KB)

 Trainable params: 881 (3.44 KB)

 Non-trainable params: 0 (0.00 B)

In [96]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report


In [97]:
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(weights))
print(class_weight_dict)

{0: np.float64(0.5634790341336616), 1: np.float64(4.438308189655173)}


In [98]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_auc", patience=5, mode="max", restore_best_weights=True
)

In [99]:
tuner = kt.RandomSearch(
    build_model,
    objective=kt.Objective("val_auc", direction="max"),
    max_trials=10,
    executions_per_trial=1,
    directory="keras_tuner_dir",
    project_name="bank_marketing_ann"
)

In [101]:
tuner.search(
    x_train_preprocessed, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    class_weight=class_weight_dict,
    verbose=1
)

Trial 10 Complete [00h 00m 42s]
val_auc: 0.7976264953613281

Best val_auc So Far: 0.8015692234039307
Total elapsed time: 00h 05m 57s


In [102]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

In [104]:
best_hp.values

{'num_layers': 1,
 'unit_0': 96,
 'activation_0': 'relu',
 'dropout_0': 0.1,
 'learning_rate': 0.01,
 'optimizer': 'nadam',
 'unit_1': 96,
 'activation_1': 'sigmoid',
 'dropout_1': 0.2}

In [105]:
best_model = tuner.hypermodel.build(best_hp)

In [106]:
history = best_model.fit(
    x_train_preprocessed, y_train,
    epochs=200,
    batch_size=64,
    validation_split=0.2,
    verbose=1,
    callbacks=[early_stop],
    class_weight=class_weight_dict,
)


Epoch 1/200
412/412 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.8000 - auc: 0.7765 - loss: 0.5562 - val_accuracy: 0.7700 - val_auc: 0.7975 - val_loss: 0.5900
Epoch 2/200
412/412 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8197 - auc: 0.7860 - loss: 0.5442 - val_accuracy: 0.8514 - val_auc: 0.7870 - val_loss: 0.4603
Epoch 3/200
412/412 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8260 - auc: 0.7867 - loss: 0.5422 - val_accuracy: 0.8401 - val_auc: 0.7945 - val_loss: 0.4720
Epoch 4/200
412/412 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8260 - auc: 0.7909 - loss: 0.5376 - val_accuracy: 0.8284 - val_auc: 0.7984 - val_loss: 0.5110
Epoch 5/200
412/412 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8285 - auc: 0.7976 - loss: 0.5339 - val_accuracy: 0.8464 - val_auc: 0.7936 - val_loss: 0.4964
Epoch 6/200
412/412 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8275 - auc: 0.7987 - loss: 0.5328 - val_accuracy: 0.8361 - val_auc: 0.7953 - val_loss: 0.4908
Epoch 7/200
412/412 ━━━━━━━━━━━━━

In [108]:
from sklearn.metrics import  roc_auc_score

y_proba = best_model.predict(x_test_preprocessed).ravel()
y_pred = (y_proba > 0.5).astype(int)

print(classification_report(y_test, y_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))

258/258 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
              precision    recall  f1-score   support

           0       0.95      0.86      0.91      7310
           1       0.38      0.66      0.48       928

    accuracy                           0.84      8238
   macro avg       0.67      0.76      0.69      8238
weighted avg       0.89      0.84      0.86      8238

Test ROC-AUC: 0.7994798545921977


In [111]:
from xgboost import XGBClassifier

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=neg / pos, random_state=42, n_jobs=-1
)
xgb.fit(x_train_preprocessed, y_train)

xgb_proba = xgb.predict_proba(x_test_preprocessed)[:, 1]
print("XGB ROC-AUC:", roc_auc_score(y_test, xgb_proba))
print(classification_report(y_test, (xgb_proba > 0.5).astype(int)))

XGB ROC-AUC: 0.8108659311759989
              precision    recall  f1-score   support

           0       0.95      0.87      0.91      7310
           1       0.38      0.65      0.48       928

    accuracy                           0.84      8238
   macro avg       0.67      0.76      0.69      8238
weighted avg       0.89      0.84      0.86      8238



In [112]:
import joblib
best_model.save("bank_ann_model.keras")
joblib.dump(preprocessing, "bank_preprocessing.pkl")
joblib.dump(xgb, "bank_xgb_model.pkl")

['bank_xgb_model.pkl']